# AutoQA Test Manual

## Table of Contents
- [Quick Start](#quick-start)
- [Environment Setup](#environment-setup)
- [Test Inventory](#test-inventory)
- [Fixture Reference](#fixture-reference)
- [Data Structure Reference](#data-structure-reference)
- [Running Tests](#running-tests)
- [Troubleshooting](#troubleshooting)
- [CI/CD Integration](#cicd-integration)
- [Maintenance](#maintenance)

---

## Quick Start

### Run All Tests
```bash
pytest tests/
```

### Run Unit Tests Only
```bash
pytest tests/unit/
```

### Run Integration Tests Only
```bash
pytest tests/integration/ -m integration
```

### Run API Tests Only
```bash
pytest tests/api/
```

### Run Specific Pipeline Tests
```bash
# RTM (test_suite_reviewer)
pytest tests/unit/test_suite_reviewer/
pytest tests/integration/test_suite_reviewer/

# Test Case Reviewer
pytest tests/unit/test_case_reviewer/
pytest tests/integration/test_case_reviewer/

# Hazard Risk Reviewer
pytest tests/unit/hazard_risk_reviewer/
pytest tests/integration/hazard_risk_reviewer/
```

### Run with Coverage Report
```bash
pytest tests/ --cov=autoqa --cov-report=html
open htmlcov/index.html
```

### Run Specific Test by Name
```bash
pytest tests/unit/test_suite_reviewer/test_synthesizer_node.py::test_synthesizer_all_yes -v
```

---

## Environment Setup

### Required Environment Variables

#### Integration Tests
Integration tests require a real LLM API connection. Set these variables before running:

```bash
export PYTEST_API_KEY="your-openai-api-key"
export PYTEST_BASE_URL="https://api.openai.com/v1"  # Optional, defaults to OpenAI
export PYTEST_MODEL="gpt-4"  # Or "claude-3-sonnet-20240229", etc.
```

**Security Note**: Integration tests validate that `PYTEST_BASE_URL` does not contain "prod" to prevent accidental use of production endpoints.

#### Performance Tuning
```bash
export AUTOQA_FANOUT_CONCURRENCY=5  # Max concurrent LLM calls (default: 5)
```

**Recommended values**:
- **Local development**: 3-5 (avoid rate limit exhaustion)
- **CI (GitHub Actions)**: 2-3 (shared runner resources)
- **Dedicated test infra**: 10-20 (if RPM/TPM headroom allows)

#### API Tests
API tests use in-process testing (no separate server needed), but you can optionally test against a running server:

```bash
# Optional: Start the API server in a separate terminal
uvicorn autoqa.api.main:app --reload --port 8000

# Run API tests
pytest tests/api/
```

---

## Test Inventory

### Unit Tests

#### Test Suite Reviewer (RTM)

| Test File | Test Function | Fixture | Runtime | Purpose |
|-----------|---------------|---------|---------|---------|
| `test_decomposer_node.py` | `test_call_with_valid_input` | `decomposer_cases.jsonl` | <1s | Verify decomposer parses JSON |
| `test_decomposer_node.py` | `test_call_missing_requirement` | - | <1s | Verify graceful failure on missing input |
| `test_decomposer_node.py` | `test_call_invalid_json` | - | <1s | Verify error handling for malformed JSON |
| `test_summary_node.py` | `test_call_with_valid_input` | `summarizer_cases.jsonl` | <1s | Verify summarizer parses JSON |
| `test_summary_node.py` | `test_call_missing_test_cases` | - | <1s | Verify graceful failure on missing input |
| `test_synthesizer_node.py` | `test_synthesizer_all_yes` | inline mock | <1s | Verify M1-M5 all-Yes verdict |
| `test_synthesizer_node.py` | `test_synthesizer_m2_na_still_yes` | inline mock | <1s | Verify N-A handling (M2, M3) |
| `test_synthesizer_node.py` | `test_synthesizer_m1_no_drives_overall_no` | inline mock | <1s | Verify No verdict propagation |
| `test_synthesizer_node.py` | `test_synthesizer_partial_yes_allowed` | inline mock | <1s | Verify partial flag validation |
| `test_synthesizer_node.py` | `test_synthesizer_skip_on_missing_state` | - | <1s | Verify skip on missing state |
| `test_synthesizer_node.py` | `test_synthesizer_invalid_json_returns_none` | - | <1s | Verify error handling for malformed JSON |

**Total RTM Unit Tests**: 11 tests, ~5 seconds total runtime

#### Test Case Reviewer

| Test File | Test Function | Fixture | Runtime | Purpose |
|-----------|---------------|---------|---------|---------|
| `test_tc_reviewer_nodes.py` | `test_tc_decomposer_validate_state_empty` | - | <1s | Verify state validation |
| `test_tc_reviewer_nodes.py` | `test_tc_decomposer_loops_over_requirements` | inline mock | <1s | Verify sequential loop over requirements |
| `test_tc_reviewer_nodes.py` | `test_tc_decomposer_skip_no_requirements` | - | <1s | Verify skip on empty requirements |
| `test_tc_reviewer_nodes.py` | `test_tc_decomposer_inner_failure_returns_none` | - | <1s | Verify error handling |
| `test_tc_reviewer_nodes.py` | `test_coverage_node_returns_specanalysis` | inline mock | <1s | Verify per-spec coverage analysis |
| `test_tc_reviewer_nodes.py` | `test_coverage_node_skip_on_missing_state` | - | <1s | Verify skip on missing state |
| `test_tc_reviewer_nodes.py` | `test_coverage_node_invalid_json_returns_empty` | - | <1s | Verify error handling |
| `test_tc_reviewer_nodes.py` | `test_overall_axis_node_returns_overallanalysis` | inline mock | <1s | Verify logical/prereqs analysis (parametrized) |
| `test_tc_reviewer_nodes.py` | `test_overall_axis_node_skip_on_missing_state` | - | <1s | Verify skip on missing state (parametrized) |
| `test_tc_reviewer_nodes.py` | `test_dispatch_coverage_emits_one_send_per_spec` | - | <1s | Verify Send fan-out dispatcher |
| `test_tc_reviewer_nodes.py` | `test_dispatch_coverage_empty_state` | - | <1s | Verify dispatcher handles empty state |
| `test_tc_reviewer_nodes.py` | `test_aggregator_returns_assessment` | inline mock | <1s | Verify 5-objective checklist aggregation |
| `test_tc_reviewer_nodes.py` | `test_aggregator_skip_on_missing_state` | - | <1s | Verify skip on missing state |
| `test_tc_reviewer_nodes.py` | `test_load_default_review_objectives` | - | <1s | Verify review objectives loader |

**Total TC Unit Tests**: 14 tests, ~7 seconds total runtime

#### Hazard Risk Reviewer

| Test File | Test Function | Fixture | Runtime | Purpose |
|-----------|---------------|---------|---------|---------|
| `test_hazard_risk_review_nodes.py` | (various) | - | <1s each | Verify H1-H7 evaluator nodes |

**Total Hazard Unit Tests**: ~10 tests, ~5 seconds total runtime

#### Common Utilities

| Test File | Test Function | Fixture | Runtime | Purpose |
|-----------|---------------|---------|---------|---------|
| `test_json_extraction.py` | (various) | - | <1s each | Verify JSON extraction utilities |

**Total Common Unit Tests**: ~5 tests, ~2 seconds total runtime

---

### Integration Tests

#### Test Suite Reviewer (RTM)

| Test File | Test Function | Fixture | Runtime | Purpose |
|-----------|---------------|---------|---------|---------|
| `test_decomposer_node.py` | `test_decomposer_node_happy_path` | - | ~5s | Verify decomposer with real LLM |
| `test_summary_node.py` | `test_summary_node_happy_path` | - | ~5s | Verify summarizer with real LLM |
| `test_coverage_evaluator_node.py` | `test_coverage_evaluator_node_happy_path` | - | ~10s | Verify coverage evaluator with real LLM |
| `test_pipeline.py` | `test_pipeline_decomposer_node` | - | ~5s | Verify decomposer in full pipeline |
| `test_pipeline.py` | `test_pipeline_summarizer_node` | - | ~5s | Verify summarizer in full pipeline |
| `test_pipeline.py` | `test_pipeline_coverage_node` | - | ~10s | Verify coverage evaluator in full pipeline |
| `test_pipeline.py` | `test_pipeline_full_state` | - | ~15s | End-to-end pipeline validation |
| `test_pipeline.py` | `test_pipeline_parametrized_fanout` | `vl.jsonl` | ~2min | Batch run with default prompts |
| `test_pipeline.py` | `test_pipeline_parametrized_standard_coverage_fanout` | `vl.jsonl` | ~2min | Batch run with v4-v7 prompts |
| `test_pipeline.py` | `test_pipeline_parametrized_advanced_coverage_fanout` | `vl.jsonl` | ~2min | Batch run with v2-v4 prompts |

**Total RTM Integration Tests**: 10 tests, ~7 minutes total runtime (batch tests run in parallel)

#### Test Case Reviewer

| Test File | Test Function | Fixture | Runtime | Purpose |
|-----------|---------------|---------|---------|---------|
| `test_tc_pipeline.py` | `test_tc_pipeline_parametrized_fanout` | `converted_PRJ01624_VL_1.1.000_tc_reviewer_format.jsonl` | ~3min | Batch TC review with real LLM |

**Total TC Integration Tests**: 1 test, ~3 minutes runtime

#### Hazard Risk Reviewer

| Test File | Test Function | Fixture | Runtime | Purpose |
|-----------|---------------|---------|---------|---------|
| `test_hazard_pipeline.py` | `test_hazard_pipeline_full_state` | `sample_hazard.json` | ~30s | End-to-end hazard review |
| `test_hazard_pipeline.py` | `test_hazard_pipeline_parallelism_verification` | `sample_hazard.json` | ~30s | Verify parallel execution topology |

**Total Hazard Integration Tests**: 2 tests, ~1 minute total runtime

---

### API Tests

| Test File | Test Function | Fixture | Runtime | Purpose |
|-----------|---------------|---------|---------|---------|
| `test_routes.py` | `test_health_check` | - | <1s | Verify /health endpoint |
| `test_routes.py` | `test_rtm_review_happy_path` | - | ~15s | Verify /api/v1/review happy path |
| `test_routes.py` | `test_rtm_review_missing_requirement` | - | <1s | Verify 422 on missing requirement |
| `test_routes.py` | `test_rtm_review_empty_test_cases` | - | <1s | Verify 422 on empty test cases |
| `test_routes.py` | `test_rtm_review_missing_thread_id` | - | <1s | Verify 422 on missing thread_id |
| `test_routes.py` | `test_rtm_review_invalid_requirement_structure` | - | <1s | Verify 422 on malformed requirement |
| `test_routes.py` | `test_tc_review_happy_path` | - | ~20s | Verify /api/v1/test-case-review happy path |
| `test_routes.py` | `test_tc_review_missing_test_case` | - | <1s | Verify 422 on missing test case |
| `test_routes.py` | `test_tc_review_empty_requirements` | - | <1s | Verify 422 on empty requirements |
| `test_routes.py` | `test_hazard_review_happy_path` | `sample_hazard.json` | ~30s | Verify /api/v1/hazard-review happy path |
| `test_routes.py` | `test_hazard_review_missing_hazard` | - | <1s | Verify 422 on missing hazard |
| `test_routes.py` | `test_hazard_review_invalid_hazard_structure` | - | <1s | Verify 422 on malformed hazard |
| `test_routes.py` | `test_invalid_json_body` | - | <1s | Verify 422 on invalid JSON |
| `test_routes.py` | `test_request_id_header_present` | - | ~15s | Verify X-Request-ID header |
| `test_routes.py` | `test_concurrent_rtm_reviews` | - | ~1min | Verify concurrent request handling |

**Total API Tests**: 15 tests, ~3 minutes total runtime

---

## Fixture Reference

### Mock Fixtures (`tests/fixtures/mock/`)

Mock LLM responses for unit tests. Each fixture follows the schema:
```json
{
  "id": "unique-test-case-id",
  "mock_response": "{...JSON response from LLM...}",
  "input_state": {...state passed to node...},
  "expected": {...expected output assertions...}
}
```

| File | Purpose | Schema | Used By |
|------|---------|--------|---------|
| `decomposer_cases.jsonl` | Mock LLM responses for decomposer | `{id, mock_response, input_state, expected}` | `test_decomposer_node.py` |
| `summarizer_cases.jsonl` | Mock LLM responses for summarizer | `{id, mock_response, input_state, expected}` | `test_summary_node.py` |
| `coverage_evaluator_cases.jsonl` | Mock LLM responses for coverage evaluator | `{id, mock_response, input_state, expected}` | `test_coverage_evaluator_node.py` |
| `synthesizer_cases.jsonl` | Mock LLM responses for synthesizer | `{id, mock_response, input_state, expected}` | `test_synthesizer_node.py` (TODO: parametrized test) |
| `tc_aggregator_cases.jsonl` | Mock LLM responses for TC aggregator | `{id, mock_response, input_state, expected}` | `test_tc_reviewer_nodes.py` (TODO: parametrized test) |

**Note**: `generator_cases.jsonl` is legacy and not currently used. It contains mock responses for an AI test case generator node that may be implemented in the future.

### Gold Fixtures (`tests/fixtures/gold/`)

Canonical labeled datasets for ML evaluation and ground-truth testing.

| File | Purpose | Schema | Used By |
|------|---------|--------|---------|
| `gold_dataset.jsonl` | Labeled RTM evaluation dataset | `{requirement, test_cases, expected_verdict}` | MLflow evaluation, future parametrized tests |
| `gold_dataset-tc.jsonl` | Labeled TC evaluation dataset | `{test_case, upstream_requirements, expected_verdict, expected_partial_objectives, primary_failure}` | MLflow evaluation, TC pipeline tests |
| `gold_dataset_labeled.jsonl` | Labeled variant with annotations | (same as gold_dataset.jsonl with additional metadata) | MLflow evaluation |
| `api_test_inputs.jsonl` | API endpoint test payloads | `{endpoint, payload}` | `test_routes.py` (future parametrized tests) |

### Local Fixtures (`tests/fixtures/local/`)

Project-specific converted/derived fixtures from real projects.

| File | Purpose | Schema | Used By |
|------|---------|--------|---------|
| `vl.jsonl` | VL project RTM inputs | `{requirement, test_cases}` | `test_pipeline.py` (batch tests) |
| `vl-1.jsonl` | VL project subset (1 record) | `{requirement, test_cases}` | Manual testing |
| `converted_PRJ01624_VL_1.1.000_tc_reviewer_format.jsonl` | VL project TC inputs | `{test_case, upstream_requirements, expected_overall_verdict, expected_partial_objectives, primary_failure, description}` | `test_tc_pipeline.py` |
| `inputs_PRJ01624_-_VL_1.1.000_Patient_Safety_Risk_-_LLM_Test_Case_-_Review.jsonl` | VL project raw JAMA export | (JAMA-specific schema) | Conversion scripts |
| `jama_reviews.jsonl` | JAMA review data | (JAMA-specific schema) | Analysis scripts |

### External Fixtures (`tests/fixtures/external/`)

Third-party or reference datasets that shouldn't be modified.

| File | Purpose | Schema | Used By |
|------|---------|--------|---------|
| `sample_hazard.json` | Canonical hazard record | `HazardRecord` (ISO 14971 / IEC 62304) | `test_hazard_pipeline.py`, `conftest.py::sample_hazard` |
| `hc_pipeline_inputs.jsonl` | Third-party HC dataset | `{requirement, test_cases}` | Reference/comparison |
| `pytest_dataset_test_suite.jsonl` | Third-party pytest dataset | (pytest-specific schema) | Reference/comparison |

### MLflow Evaluation Fixtures

| Directory | Purpose | Schema | Used By |
|-----------|---------|--------|---------|
| `mlflow_eval/` | Full MLflow evaluation datasets | `eval_inputs.jsonl`, `eval_outputs.jsonl`, `eval_outputs_labels.jsonl` | MLflow evaluation scripts |
| `mlflow_eval_quick/` | Quick MLflow smoke tests | (same as mlflow_eval/) | CI smoke tests |

### Generated Fixtures (`tests/fixtures/generated/`)

Runtime outputs from batch runs. **Do not commit these to version control** (except `.gitkeep`).

| File Pattern | Purpose | Generated By |
|--------------|---------|--------------|
| `inputs_batch_*.jsonl` | Batch input records | Batch run scripts |
| `outputs_batch_*.jsonl` | Batch output records | Batch run scripts |
| `batch_*_log.txt` | Batch run logs | Batch run scripts |
| `verification_report.txt` | Batch verification report | Verification scripts |

---

## Data Structure Reference

This section documents the input and output data structures for each reviewer type, with real examples from the test fixtures.

### RTM (Test Suite Reviewer)

The RTM reviewer evaluates whether a set of test cases provides adequate coverage of a software requirement across functional, negative, and boundary dimensions.

#### Input Structure (Minimal)

```json
{
  "requirement": {
    "req_id": "REQ-HC-001",
    "text": "System shall allow providers to prescribe medications electronically and select the patient's preferred pharmacy from a nationwide directory."
  },
  "test_cases": [
    {
      "test_id": "TC-HC-001-A",
      "description": "Verify provider can electronically prescribe a medication successfully.",
      "setup": "EHR test environment; test provider logged in; test patient chart open.",
      "steps": "Step: 1. Navigate to Orders -> Medications.\nStep: 2. Search and select Amoxicillin.\nStep: 3. Enter dosage and frequency.\nStep: 4. Click 'Sign & Send'.",
      "expectedResults": "ExpectedResult: 1. Orders page opens with Medications section available.\nExpectedResult: 2. Amoxicillin is successfully selected and displayed as the intended medication.\nExpectedResult: 3. Medication order with entered dosage and frequency is saved to the patient record.\nExpectedResult: 4. Order status updates to 'Sent' in the clinical workflow."
    }
  ]
}
```

**Fixture Example**: `external/hc_pipeline_inputs.jsonl`

#### Input Structure (With Optional Design Docs)

```json
{
  "requirement": {
    "req_id": "REQ-PUMP-SW-042",
    "text": "The infusion pump software shall implement a rate-limiting algorithm that constrains the commanded flow rate to a maximum of 1200 mL/hr and a minimum of 0.1 mL/hr, with a resolution of 0.1 mL/hr increments."
  },
  "test_cases": [ /* ... */ ],
  "design_docs": [  // ⚠️ OPTIONAL - provides design context
    {
      "doc_id": "DD-PUMP-ALGO-001",
      "name": "Rate Limiting Algorithm Design",
      "description": "Describes the PID controller implementation, saturation limits, and anti-windup logic for the rate-limiting control loop. Specifies the input validation logic that enforces minimum (0.1 mL/hr), maximum (1200.0 mL/hr), and resolution (0.1 mL/hr increments) constraints before commanding the motor driver."
    },
    {
      "doc_id": "IDD-PUMP-MOTOR-001",
      "name": "Motor Driver Interface Design",
      "description": "Specifies the hardware abstraction layer for motor control signals, pulse timing constraints, and safe-state latch behavior. Documents the pulse-width modulation (PWM) frequency and duty cycle calculations that translate commanded flow rates into motor driver signals."
    }
  ]
}
```

**Fixture Example**: `external/rtm_with_design_docs.jsonl` ⭐

**Key Input Fields:**
- `requirement.req_id` (optional): Unique requirement identifier
- `requirement.text`: The requirement statement to evaluate
- `test_cases[]`: Array of test cases traced to this requirement
  - `test_id`: Unique test case identifier
  - `description`: What the test verifies
  - `setup`: Preconditions and test environment
  - `steps`: Numbered test execution steps
  - `expectedResults`: Numbered expected outcomes
- `design_docs[]` (optional): Design documents providing architectural context
  - `doc_id`: Unique design document identifier
  - `name`: Design document title
  - `description`: Design document description

#### Output Structure

```json
{
  "decomposed_requirement": {
    "requirement": { "req_id": "REQ-HC-001", "text": "..." },
    "decomposed_specifications": [
      {
        "spec_id": "REQ-HC-001-S1",
        "description": "System shall allow providers to prescribe medications electronically",
        "acceptance_criteria": "Provider can search formulary, enter dose/frequency, and submit prescription",
        "rationale": "Core e-prescribing functionality"
      },
      {
        "spec_id": "REQ-HC-001-S2",
        "description": "System shall allow providers to select patient's preferred pharmacy",
        "acceptance_criteria": "Provider can search and select from nationwide pharmacy directory",
        "rationale": "Pharmacy routing requirement"
      }
    ]
  },
  "test_suite": {
    "requirement": { "req_id": "REQ-HC-001", "text": "..." },
    "test_cases": [ /* original test cases */ ],
    "summary": [
      {
        "test_case_id": "TC-HC-001-A",
        "objective": "Verify electronic prescription submission",
        "verifies": "Medication order creation and transmission",
        "protocol": ["Navigate to Orders", "Select medication", "Enter details", "Submit"],
        "acceptance_criteria": ["Order saved", "Status updated to Sent"],
        "is_generated": false
      }
    ]
  },
  "coverage_analysis": [
    {
      "spec_id": "REQ-HC-001-S1",
      "covered_exists": true,
      "covered_by_test_cases": [
        {
          "test_case_id": "TC-HC-001-A",
          "dimensions": ["functional"],
          "rationale": "Verifies core prescription submission flow"
        }
      ]
    },
    {
      "spec_id": "REQ-HC-001-S2",
      "covered_exists": false,
      "covered_by_test_cases": []
    }
  ],
  "synthesized_assessment": {
    "requirement": { "req_id": "REQ-HC-001", "text": "..." },
    "overall_verdict": "No",
    "mandatory_findings": [
      {
        "code": "M1",
        "dimension": "Functional",
        "verdict": "Yes",
        "partial": false,
        "rationale": "TC-HC-001-A verifies electronic prescription submission",
        "cited_test_case_ids": ["TC-HC-001-A"],
        "uncovered_spec_ids": []
      },
      {
        "code": "M2",
        "dimension": "Negative",
        "verdict": "Yes",
        "partial": false,
        "rationale": "TC-HC-001-B verifies invalid dose rejection",
        "cited_test_case_ids": ["TC-HC-001-B"],
        "uncovered_spec_ids": []
      },
      {
        "code": "M3",
        "dimension": "Boundary",
        "verdict": "N-A",
        "partial": false,
        "rationale": "No numeric thresholds or limits in this requirement",
        "cited_test_case_ids": [],
        "uncovered_spec_ids": []
      },
      {
        "code": "M4",
        "dimension": "Spec Coverage",
        "verdict": "No",
        "partial": false,
        "rationale": "Spec REQ-HC-001-S2 (pharmacy selection) has no covering test cases",
        "cited_test_case_ids": [],
        "uncovered_spec_ids": ["REQ-HC-001-S2"]
      },
      {
        "code": "M5",
        "dimension": "Terminology",
        "verdict": "Yes",
        "partial": false,
        "rationale": "Test cases use requirement vocabulary (prescribe, pharmacy, medication)",
        "cited_test_case_ids": [],
        "uncovered_spec_ids": []
      }
    ],
    "comments": "Pharmacy selection functionality is not covered by any test case.",
    "clarification_questions": [
      "Is pharmacy selection implemented in the current build?",
      "Should pharmacy selection be tested separately?"
    ]
  }
}
```

**Key Output Fields:**
- `decomposed_requirement`: Requirement broken into atomic specifications
- `test_suite.summary`: Condensed view of each test case's purpose
- `coverage_analysis[]`: Per-spec coverage verdict with dimensions (functional/negative/boundary)
- `synthesized_assessment`: Final M1-M5 rubric with overall Yes/No verdict
  - `overall_verdict`: "Yes" if all M1-M5 are Yes or N-A, otherwise "No"
  - `mandatory_findings[]`: Exactly 5 items (M1-M5) in order
  - `partial`: True when verdict=Yes but coverage is incomplete (drives Yellow UI rendering)

---

### Test Case Reviewer

The Test Case Reviewer evaluates a single test case against one or more traced requirements using a 5-objective checklist.

#### Input Structure (Minimal)

```json
{
  "test_case": {
    "test_id": "TC-HC-001-B",
    "description": "Verify the e-Prescribing module rejects an invalid dose value with a validation error and prevents transmission to the pharmacy.",
    "setup": "EHR build v9.4.2 deployed in QA environment; provider 'Dr. Lee' logged in via SSO; test patient 'Smith, John' (MRN 12345) chart open; pharmacy connector enabled.",
    "steps": "Step: 1. Navigate to Orders -> Medications.\nStep: 2. Search for 'Amoxicillin 500mg tablet' and select it.\nStep: 3. Enter '-500' into the Dose field.\nStep: 4. Click 'Sign & Send'.\nStep: 5. Inspect the Dose field state and the pharmacy outbound queue.",
    "expectedResults": "ExpectedResult: 1. The Medications page loads.\nExpectedResult: 2. 'Amoxicillin 500mg tablet' is selectable.\nExpectedResult: 3. The Dose field is highlighted red with error 'Dose must be a positive numeric value'.\nExpectedResult: 4. Order remains in 'Draft' status and pharmacy queue shows zero new entries."
  },
  "requirements": [
    {
      "req_id": "REQ-HC-002",
      "text": "Medication orders shall reject non-positive numeric dose values with an inline validation error and shall not be saved to the patient record until validation passes."
    },
    {
      "req_id": "REQ-HC-005",
      "text": "The pharmacy outbound queue shall not receive transmissions for medication orders that fail validation; orders remaining in 'Draft' status are not transmitted."
    }
  ],
  "review_objectives": [
    {
      "id": "expected_result_support",
      "description": "Each expected result is measurable, concrete, and directly observable"
    },
    {
      "id": "expected_result_spec_align",
      "description": "Expected results collectively verify all decomposed specs from traced requirements"
    },
    {
      "id": "test_case_achieves",
      "description": "The final test step verifies the intended outcome"
    },
    {
      "id": "test_case_logical_sequence",
      "description": "Steps follow a logical setup -> stimulus -> verification flow"
    },
    {
      "id": "test_case_setup_clarity",
      "description": "Setup documents all preconditions needed to reproduce the test"
    }
  ]
}
```

**Key Input Fields:**
- `test_case`: The test case to review (same structure as RTM input)
- `requirements[]`: One or more requirements traced to this test case
- `review_objectives[]` (optional): Custom checklist; defaults to standard 5 objectives if omitted



**Fixture Example**: `local/converted_PRJ01624_VL_1.1.000_tc_reviewer_format.jsonl`

#### Input Structure (With Optional Fields)

```json
{
  "test_case": { /* ... */ },
  "requirements": [ /* ... */ ],
  "review_objectives": [  // ⚠️ OPTIONAL - custom checklist
    {
      "id": "expected_result_support",
      "description": "Each expected result is measurable, concrete, and directly observable"
    },
    {
      "id": "expected_result_spec_align",
      "description": "Expected results collectively verify all decomposed specs from traced requirements"
    },
    {
      "id": "test_case_achieves",
      "description": "The final test step verifies the intended outcome"
    },
    {
      "id": "test_case_logical_sequence",
      "description": "Steps follow a logical setup -> stimulus -> verification flow"
    },
    {
      "id": "test_case_setup_clarity",
      "description": "Setup documents all preconditions needed to reproduce the test"
    }
  ],
  "design_docs": [  // ⚠️ OPTIONAL - design context
    {
      "doc_id": "DD-EHR-AUTH-MFA-001",
      "name": "Multi-Factor Authentication Architecture",
      "description": "Describes the two-phase authentication flow: first-factor (password) validation against the user database, followed by second-factor (TOTP) validation using the HMAC-based One-Time Password algorithm (RFC 6238). Documents the session state machine that tracks MFA completion status and prevents bypass via direct URL navigation."
    }
  ]
}
```

**Fixture Example**: `external/tc_with_design_docs.jsonl` ⭐

**Key Input Fields:**
- `test_case`: The test case to review (same structure as RTM input)
- `requirements[]`: One or more requirements traced to this test case
- `review_objectives[]` (optional): Custom checklist; defaults to standard 5 objectives if omitted
- `design_docs[]` (optional): Design documents providing architectural context

#### Output Structure

```json
{
  "test_case": { /* echo of input test case */ },
  "requirements": [ /* echo of input requirements */ ],
  "decomposed_requirements": [
    {
      "requirement": { "req_id": "REQ-HC-002", "text": "..." },
      "decomposed_specifications": [
        {
          "spec_id": "REQ-HC-002-S1",
          "description": "System shall reject non-positive dose values",
          "acceptance_criteria": "Dose field validation triggers on negative or zero input",
          "rationale": "Input validation requirement"
        },
        {
          "spec_id": "REQ-HC-002-S2",
          "description": "System shall display inline validation error",
          "acceptance_criteria": "Error message appears beneath dose field",
          "rationale": "User feedback requirement"
        }
      ]
    }
  ],
  "coverage_analysis": [
    {
      "spec_id": "REQ-HC-002-S1",
      "exists": true,
      "assessment": "ExpectedResult 3 verifies dose field validation triggers on '-500' input"
    },
    {
      "spec_id": "REQ-HC-002-S2",
      "exists": true,
      "assessment": "ExpectedResult 3 verifies inline error message 'Dose must be a positive numeric value'"
    }
  ],
  "logical_structure_analysis": {
    "exists": true,
    "assessment": "Steps follow setup (Step 1-2) -> stimulus (Step 3-4) -> verification (Step 5) flow"
  },
  "prereqs_analysis": {
    "exists": true,
    "assessment": "Setup documents EHR version, provider credentials, patient state, and pharmacy connector state"
  },
  "aggregated_assessment": {
    "test_case": { /* echo */ },
    "requirements": [ /* echo */ ],
    "decomposed_requirements": [ /* echo */ ],
    "evaluated_checklist": [
      {
        "id": "expected_result_support",
        "description": "Each expected result is measurable, concrete, and directly observable",
        "verdict": "Yes",
        "partial": false,
        "assessment": "All 4 ExpectedResults are concrete (page loads, item selectable, exact error text, exact status and queue state)"
      },
      {
        "id": "expected_result_spec_align",
        "description": "Expected results collectively verify all decomposed specs",
        "verdict": "Yes",
        "partial": false,
        "assessment": "Both REQ-HC-002 specs and both REQ-HC-005 specs are verified by ExpectedResults 3-4"
      },
      {
        "id": "test_case_achieves",
        "description": "The final test step verifies the intended outcome",
        "verdict": "Yes",
        "partial": false,
        "assessment": "Step 5 explicitly inspects dose field state and pharmacy queue, verifying the negative-test outcome"
      },
      {
        "id": "test_case_logical_sequence",
        "description": "Steps follow a logical setup -> stimulus -> verification flow",
        "verdict": "Yes",
        "partial": false,
        "assessment": "Steps 1-2 establish context, Steps 3-4 trigger validation, Step 5 verifies outcome"
      },
      {
        "id": "test_case_setup_clarity",
        "description": "Setup documents all preconditions needed to reproduce",
        "verdict": "Yes",
        "partial": false,
        "assessment": "Setup names EHR version, provider role, patient state, and pharmacy connector state"
      }
    ],
    "overall_verdict": "Yes",
    "comments": "",
    "clarification_questions": []
  }
}
```

**Key Output Fields:**
- `decomposed_requirements[]`: Each traced requirement broken into atomic specs
- `coverage_analysis[]`: Per-spec verdict (does the test case verify this spec?)
- `logical_structure_analysis`: Test-case-level verdict on logical flow
- `prereqs_analysis`: Test-case-level verdict on setup completeness
- `aggregated_assessment.evaluated_checklist[]`: 5-objective checklist with verdicts
- `aggregated_assessment.overall_verdict`: "Yes" if all 5 objectives are Yes, otherwise "No"
- `partial`: True when verdict=Yes but coverage is materially incomplete

---

### Hazard Risk Reviewer

The Hazard Risk Reviewer evaluates whether traced requirements and test cases provide reasonable assurance of safety against a hazard, applying the H1-H7 rubric from ISO 14971 / IEC 62304.

#### Input Structure (Minimal)

```json
{
  "hazard_id": "HAZ-PUMP-001",
  "hazardous_situation_id": "HS-PUMP-001",
  "hazard": "Over-infusion of medication due to software loop hang",
  "hazardous_situation": "Patient receives medication at maximum pump rate continuously, exceeding prescribed dose, while pump UI fails to indicate runaway condition.",
  "function": "Continuous infusion rate control loop",
  "ots_software": "FreeRTOS 10.4.3",
  "hazardous_sequence_of_events": "1. Periodic timer ISR fails to fire due to scheduler stall. 2. Rate-control loop continues to issue motor pulses at most-recently-commanded rate. 3. UI thread continues to display nominal infusion-rate. 4. Pump delivers medication at maximum rate until manually halted.",
  "software_related_causes": "Scheduler stall under heavy task load; missing independent watchdog on rate-control loop; UI thread not gated on heartbeat from rate-control task.",
  "harm_severity_rationale": "External risk controls reduce but do not eliminate chance of clinically significant over-infusion before detection.",
  "harm": "Severe over-infusion with potential for life-threatening overdose (insulin shock, cytotoxicity).",
  "severity": "Catastrophic",
  "exploitability_pre_mitigation": "Not applicable (not a cyber-exploitable surface)",
  "probability_of_harm_pre_mitigation": "Probable",
  "initial_risk_rating": "Unacceptable",
  "risk_control_measures": "REQ-PUMP-101 mandates rate-control loop monitored by independent hardware watchdog that latches motor driver into safe state if heartbeats missed for >200ms. REQ-PUMP-102 mandates UI thread gated on same heartbeat and renders Alarm Mode banner when heartbeats stop.",
  "demonstration_of_effectiveness": "Verified by TC-PUMP-201 (functional heartbeat), TC-PUMP-202 (fault injection — scheduler stall), TC-PUMP-203 (boundary — heartbeat exactly at 200ms latency).",
  "severity_of_harm_post_mitigation": "Catastrophic",
  "exploitability_post_mitigation": "Not applicable",
  "probability_of_harm_post_mitigation": "Remote",
  "final_risk_rating": "Acceptable",
  "new_hs_reference": "",
  "sw_fmea_trace": "FMEA-PUMP-RC-001",
  "sra_link": "SRA-PUMP-2025-12",
  "urra_item": "URRA-PUMP-RC-001",
  "residual_risk_acceptability": "Per GQP-10-02 Risk Management Report, residual risk is acceptable: hardware watchdog and UI alarm provide redundant detection and shutoff, post-mitigation probability is Remote with verified detection latency under 200ms.",
  "requirements": [
    {
      "req_id": "REQ-PUMP-101",
      "text": "The rate-control loop shall be monitored by an independent hardware watchdog. Motor driver shall be latched into safe (no-pulse) state if heartbeat from rate-control task is missed for more than 200ms."
    },
    {
      "req_id": "REQ-PUMP-102",
      "text": "The UI thread shall render an Alarm Mode banner and silence nominal-rate displays when rate-control task heartbeat is missed for more than 200ms."
    }
  ],
  "test_cases": [
    {
      "test_id": "TC-PUMP-201",
      "description": "Functional verification of watchdog heartbeat reception under nominal load",
      "setup": "Pump in standard infusion mode; rate-control task running at 50ms cadence; hardware watchdog enabled with 200ms latch threshold.",
      "steps": "1. Start 30-minute infusion at 5 mL/hr. 2. Monitor watchdog heartbeat counter. 3. Verify motor driver remains armed and pump delivers commanded rate.",
      "expectedResults": "1. Heartbeat counter increments at expected cadence. 2. Motor driver remains armed throughout. 3. Delivered volume matches commanded volume within tolerance."
    },
    {
      "test_id": "TC-PUMP-202",
      "description": "Fault injection — simulate scheduler stall and verify watchdog latches motor driver",
      "setup": "Pump in standard infusion mode; instrumented build with debug hook to suspend rate-control task indefinitely.",
      "steps": "1. Start infusion at 5 mL/hr. 2. Trigger debug hook to suspend rate-control task. 3. Observe watchdog timer and motor driver state. 4. Observe UI banner.",
      "expectedResults": "1. Watchdog counter stalls. 2. After 200ms, motor driver latches to safe state and stops issuing pulses. 3. UI renders Alarm Mode banner and silences nominal-rate display."
    },
    {
      "test_id": "TC-PUMP-203",
      "description": "Boundary — heartbeat latency exactly at 200ms threshold",
      "setup": "Pump in standard infusion mode; instrumented build with debug hook to delay rate-control task heartbeat by configurable amount.",
      "steps": "1. Start infusion. 2. Configure debug hook to delay heartbeats by 199ms; verify motor driver remains armed. 3. Configure to 200ms; verify armed at boundary. 4. Configure to 201ms; verify motor driver latches to safe state.",
      "expectedResults": "1. At 199ms delay motor driver is armed. 2. At 200ms delay motor driver is armed (inclusive boundary). 3. At 201ms delay motor driver latches to safe state and UI Alarm Mode banner renders."
    }
  ],
  "design_docs": [
    {
      "doc_id": "DD-PUMP-RC-001",
      "name": "Rate Control Loop and Watchdog Architecture",
      "description": "Describes segregation of rate-control task from UI task, heartbeat protocol, hardware watchdog wiring, and safe-state latch behavior of motor driver."
    }
  ]
}
```

**Key Input Fields:**
- ISO 14971 hazard register fields: `hazard`, `hazardous_situation`, `harm`, `severity`, `probability_of_harm_*`, `risk_rating`
- `software_related_causes`: Software failure modes contributing to hazard
- `risk_control_measures`: Mitigation requirements (references `requirements[]`)
- `demonstration_of_effectiveness`: Verification approach (references `test_cases[]`)
- `requirements[]`: Traced requirements implementing risk controls
- `test_cases[]`: Traced test cases verifying risk controls
- `design_docs[]`: Traced design documents describing architecture



**Fixture Example**: `external/sample_hazard.json`

#### Input Structure (Full V-Model Traceability)

```json
{
  "hazard_id": "HAZ-PUMP-001",
  /* ... all ISO 14971 fields ... */,
  "user_needs": [  // ⚠️ OPTIONAL - high-level user requirements
    {
      "req_id": "UN-PUMP-003",
      "text": "The infusion system shall prevent over-infusion that could harm the patient."
    },
    {
      "req_id": "UN-PUMP-007",
      "text": "The infusion system shall provide clear and immediate feedback to clinicians when a safety-critical failure is detected."
    }
  ],
  "system_requirements": [  // ⚠️ OPTIONAL - system-level requirements
    {
      "req_id": "SYS-PUMP-015",
      "text": "The infusion system shall include independent hardware and software watchdog mechanisms to detect and respond to software failures within 200 ms."
    },
    {
      "req_id": "SYS-PUMP-016",
      "text": "The infusion system shall latch the motor driver into a safe (no-pulse) state when a software failure is detected by the watchdog mechanism."
    }
  ],
  "requirements": [  // Software requirements (always required)
    {
      "req_id": "REQ-PUMP-101",
      "text": "The rate-control loop shall be monitored by an independent hardware watchdog. Motor driver shall be latched into safe (no-pulse) state if heartbeat from rate-control task is missed for more than 200ms."
    }
  ],
  "test_cases": [ /* ... */ ],
  "design_docs": [  // ⚠️ OPTIONAL - design artifacts
    {
      "doc_id": "DD-PUMP-RC-001",
      "name": "Rate Control Loop and Watchdog Architecture",
      "description": "Describes segregation of rate-control task from UI task, heartbeat protocol, hardware watchdog wiring, and safe-state latch behavior of motor driver."
    },
    {
      "doc_id": "SFMEA-PUMP-001",
      "name": "Software FMEA - Rate Control Loop",
      "description": "Failure modes and effects analysis for the rate control software component, including scheduler stall scenarios, watchdog timeout analysis, and motor driver fail-safe behavior."
    },
    {
      "doc_id": "SRA-PUMP-2025-12",
      "name": "Software Risk Analysis - Infusion Pump v3.2",
      "description": "Comprehensive software risk analysis document that traces hazards from the hazard register to software requirements, design elements, and verification test cases."
    }
  ]
}
```

**Fixture Example**: `external/hazard_full_traceability.json` ⭐

**Key Input Fields:**
- ISO 14971 hazard register fields: `hazard`, `hazardous_situation`, `harm`, `severity`, `probability_of_harm_*`, `risk_rating`
- `software_related_causes`: Software failure modes contributing to hazard
- `risk_control_measures`: Mitigation requirements (references `requirements[]`)
- `demonstration_of_effectiveness`: Verification approach (references `test_cases[]`)
- `requirements[]`: Traced software requirements implementing risk controls (required, min 1)
- `test_cases[]`: Traced test cases verifying risk controls
- `design_docs[]` (optional): Traced design documents describing architecture
- `user_needs[]` (optional): High-level user requirements for V-model traceability
- `system_requirements[]` (optional): System-level requirements for V-model traceability

#### Output Structure

```json
{
  "hazard": { /* echo of input HazardRecord */ },
  "requirement_reviews": [
    {
      "requirement": { "req_id": "REQ-PUMP-101", "text": "..." },
      "synthesized_assessment": {
        "requirement": { "req_id": "REQ-PUMP-101", "text": "..." },
        "overall_verdict": "Yes",
        "mandatory_findings": [ /* M1-M5 rubric for this requirement */ ],
        "comments": "",
        "clarification_questions": []
      },
      "decomposed_requirement": { /* decomposed specs */ },
      "test_suite": { /* summarized test cases */ },
      "coverage_analysis": [ /* per-spec coverage */ ]
    },
    {
      "requirement": { "req_id": "REQ-PUMP-102", "text": "..." },
      "synthesized_assessment": { /* M1-M5 for REQ-PUMP-102 */ }
    }
  ],
  "hazard_assessment": {
    "hazard_id": "HAZ-PUMP-001",
    "overall_verdict": "Yes",
    "mandatory_findings": [
      {
        "code": "H1",
        "dimension": "Hazard Record Completeness and Semantic Integrity",
        "verdict": "Yes",
        "rationale": "All ISO 14971 fields populated; FSOE is step-by-step; software_related_causes names specific failure modes",
        "cited_req_ids": [],
        "cited_test_case_ids": [],
        "unblocked_items": []
      },
      {
        "code": "H2",
        "dimension": "Software Contribution and Cause Coverage",
        "verdict": "Yes",
        "rationale": "software_related_causes names scheduler stall, missing watchdog, UI thread not gated; risk_control_measures addresses all three",
        "cited_req_ids": ["REQ-PUMP-101", "REQ-PUMP-102"],
        "cited_test_case_ids": [],
        "unblocked_items": []
      },
      {
        "code": "H3",
        "dimension": "Pre-Mitigation Risk and Exploitability Characterization",
        "verdict": "Yes",
        "rationale": "Pre-mitigation probability='Probable', severity='Catastrophic', initial_risk_rating='Unacceptable' with rationale",
        "cited_req_ids": [],
        "cited_test_case_ids": [],
        "unblocked_items": []
      },
      {
        "code": "H4",
        "dimension": "Risk Control Identification, Allocation, and Coverage",
        "verdict": "Yes",
        "rationale": "REQ-PUMP-101 (hardware watchdog) and REQ-PUMP-102 (UI alarm) both pass M1-M5 rubric; all software causes addressed",
        "cited_req_ids": ["REQ-PUMP-101", "REQ-PUMP-102"],
        "cited_test_case_ids": [],
        "unblocked_items": []
      },
      {
        "code": "H5",
        "dimension": "Verification Depth and Hazard-Path Effectiveness",
        "verdict": "Yes",
        "rationale": "TC-PUMP-201 (functional), TC-PUMP-202 (fault injection), TC-PUMP-203 (boundary at 200ms) verify both watchdog and UI alarm",
        "cited_req_ids": [],
        "cited_test_case_ids": ["TC-PUMP-201", "TC-PUMP-202", "TC-PUMP-203"],
        "unblocked_items": []
      },
      {
        "code": "H6",
        "dimension": "Residual Risk Closure and Acceptability Decision",
        "verdict": "Yes",
        "rationale": "Post-mitigation probability='Remote', final_risk_rating='Acceptable'; residual_risk_acceptability cites GQP-10-02 report",
        "cited_req_ids": [],
        "cited_test_case_ids": [],
        "unblocked_items": []
      },
      {
        "code": "H7",
        "dimension": "HSHA Update and Newly Identified Hazard / Hazardous Situation Capture",
        "verdict": "Yes",
        "rationale": "new_hs_reference is empty (no new hazards identified); FMEA/SRA/URRA traceability populated",
        "cited_req_ids": [],
        "cited_test_case_ids": [],
        "unblocked_items": []
      }
    ],
    "comments": "",
    "clarification_questions": []
  }
}
```

**Key Output Fields:**
- `requirement_reviews[]`: Per-requirement RTM assessment (reuses test_suite_reviewer pipeline)
  - Each requirement gets its own M1-M5 `synthesized_assessment`
- `hazard_assessment`: Aggregated H1-H7 rubric for the hazard
  - `overall_verdict`: "Yes" if all H1-H7 are Yes or N-A, otherwise "No"
  - `mandatory_findings[]`: Exactly 7 items (H1-H7) in order
  - H5 may be "N-A" when `software_related_causes` indicates no software cause
  - `cited_req_ids[]`: Requirements supporting this finding
  - `cited_test_case_ids[]`: Test cases supporting this finding
  - `unblocked_items[]`: Populated when verdict=No with specific missing/broken elements

---

### Verdict Aggregation Rules

All three reviewers follow consistent verdict aggregation logic:

#### RTM (M1-M5)
- `overall_verdict = "Yes"` ⟺ every `mandatory_findings[i].verdict ∈ {Yes, N-A}`
- Any single `verdict = "No"` ⟹ `overall_verdict = "No"`
- `partial = true` when `verdict = "Yes"` AND coverage is materially incomplete
  - Drives Yellow rendering in viewer
  - Does NOT affect `overall_verdict` (partial-Yes still passes)

#### Test Case Reviewer (5 Objectives)
- `overall_verdict = "Yes"` ⟺ every `evaluated_checklist[i].verdict = "Yes"`
- Any single `verdict = "No"` ⟹ `overall_verdict = "No"`
- `partial = true` when `verdict = "Yes"` AND objective is materially incomplete
  - Drives Yellow rendering in viewer
  - Does NOT affect `overall_verdict` (partial-Yes still passes)

#### Hazard Risk Reviewer (H1-H7)
- `overall_verdict = "Yes"` ⟺ every `mandatory_findings[i].verdict ∈ {Yes, N-A}`
- Any single `verdict = "No"` ⟹ `overall_verdict = "No"`
- Only H5 may be "N-A" (when no software-related causes exist)
- H1, H2, H3, H4, H6, H7 must be Yes or No

---

### Common Data Models

These models are shared across all three reviewers:

#### Requirement
```json
{
  "req_id": "REQ-HC-001",  // optional
  "text": "System shall allow providers to prescribe medications electronically..."
}
```

#### TestCase
```json
{
  "test_id": "TC-HC-001-A",
  "description": "Verify provider can electronically prescribe a medication successfully.",
  "setup": "EHR test environment; test provider logged in; test patient chart open.",
  "steps": "Step: 1. Navigate to Orders -> Medications.\nStep: 2. Search and select Amoxicillin...",
  "expectedResults": "ExpectedResult: 1. Orders page opens...\nExpectedResult: 2. Amoxicillin is selected..."
}
```

#### DecomposedRequirement
```json
{
  "requirement": { "req_id": "REQ-HC-001", "text": "..." },
  "decomposed_specifications": [
    {
      "spec_id": "REQ-HC-001-S1",
      "description": "System shall allow providers to prescribe medications electronically",
      "acceptance_criteria": "Provider can search formulary, enter dose/frequency, and submit prescription",
      "rationale": "Core e-prescribing functionality"
    }
  ]
}
```

---

## Running Tests

### By Test Type

```bash
# Unit tests only (fast, no LLM calls)
pytest tests/unit/ -v

# Integration tests only (slow, requires PYTEST_API_KEY)
pytest tests/integration/ -m integration -v

# API tests only
pytest tests/api/ -v

# All tests
pytest tests/ -v
```

### By Pipeline

```bash
# RTM (test_suite_reviewer)
pytest tests/unit/test_suite_reviewer/ tests/integration/test_suite_reviewer/ -v

# Test Case Reviewer
pytest tests/unit/test_case_reviewer/ tests/integration/test_case_reviewer/ -v

# Hazard Risk Reviewer
pytest tests/unit/hazard_risk_reviewer/ tests/integration/hazard_risk_reviewer/ -v
```

### By Marker

```bash
# Integration tests only
pytest -m integration

# Slow tests only
pytest -m slow

# Skip slow tests
pytest -m "not slow"
```

### With Coverage

```bash
# HTML coverage report
pytest tests/ --cov=autoqa --cov-report=html
open htmlcov/index.html

# Terminal coverage report
pytest tests/ --cov=autoqa --cov-report=term-missing

# Coverage for specific module
pytest tests/unit/test_suite_reviewer/ --cov=autoqa.components.test_suite_reviewer
```

### Parallel Execution

```bash
# Install pytest-xdist
pip install pytest-xdist

# Run tests in parallel (4 workers)
pytest tests/unit/ -n 4

# Auto-detect number of CPUs
pytest tests/unit/ -n auto
```

**Note**: Integration tests should NOT be run in parallel with pytest-xdist due to shared rate limits. Use `AUTOQA_FANOUT_CONCURRENCY` instead to control concurrency within each test.

### Verbose Output

```bash
# Show test names and outcomes
pytest tests/ -v

# Show print statements
pytest tests/ -s

# Show full diff on assertion failures
pytest tests/ -vv

# Show local variables on failure
pytest tests/ -l
```

### Filtering Tests

```bash
# Run specific test file
pytest tests/unit/test_suite_reviewer/test_synthesizer_node.py

# Run specific test function
pytest tests/unit/test_suite_reviewer/test_synthesizer_node.py::test_synthesizer_all_yes

# Run tests matching pattern
pytest tests/ -k "synthesizer"

# Run tests NOT matching pattern
pytest tests/ -k "not slow"
```

---

## Troubleshooting

### Integration Tests Fail with "PYTEST_API_KEY not set"

**Symptom**: Integration tests are skipped with message "PYTEST_API_KEY not set — skipping integration test"

**Solution**: Set environment variable:
```bash
export PYTEST_API_KEY="your-api-key"
export PYTEST_BASE_URL="https://api.openai.com/v1"  # Optional
export PYTEST_MODEL="gpt-4"
```

**Verification**:
```bash
echo $PYTEST_API_KEY
pytest tests/integration/test_suite_reviewer/test_decomposer_node.py -v
```

---

### Integration Tests Hit Rate Limits

**Symptom**: Tests fail with "Rate limit exceeded" or "429 Too Many Requests"

**Solution 1**: Reduce concurrency:
```bash
export AUTOQA_FANOUT_CONCURRENCY=2
pytest tests/integration/
```

**Solution 2**: Run tests sequentially:
```bash
pytest tests/integration/ -v  # No -n flag
```

**Solution 3**: Increase rate limits in `autoqa/core/config.py`:
```python
max_requests_per_minute: int = 100  # Increase if you have higher tier
max_tokens_per_minute: int = 200000  # Increase if you have higher tier
```

---

### Tests Fail with "Fixture not found"

**Symptom**: `FileNotFoundError: Fixture 'xyz.jsonl' not found in any of: mock/, gold/, local/, external/ or root fixtures/`

**Solution**: Check fixture search paths in `tests/helpers.py::load_jsonl()`:
1. `mock/` - Mock LLM responses for unit tests
2. `gold/` - Canonical labeled datasets
3. `local/` - Project-specific fixtures
4. `external/` - Third-party datasets
5. Root `fixtures/` - Backwards compatibility

**Verification**:
```bash
ls tests/fixtures/mock/xyz.jsonl
ls tests/fixtures/gold/xyz.jsonl
# etc.
```

---

### API Tests Fail with "Connection refused"

**Symptom**: API tests fail with connection errors

**Solution**: API tests use in-process testing by default (no server needed). If you're testing against a running server:

```bash
# Start the API server in a separate terminal
uvicorn autoqa.api.main:app --reload --port 8000

# Update test fixture to use external server
# (Modify tests/api/test_routes.py::client fixture)
```

**Default behavior**: Tests use `ASGITransport` for in-process testing, so no server is needed.

---

### Tests Fail with "Production endpoint detected"

**Symptom**: `pytest.fail: PYTEST_BASE_URL appears to be a production endpoint`

**Solution**: This is a security feature. Integration tests must use test/staging endpoints only.

```bash
# ✅ Correct
export PYTEST_BASE_URL="https://api.staging.example.com/v1"

# ❌ Incorrect (contains "prod")
export PYTEST_BASE_URL="https://api.production.example.com/v1"
```

---

### Parametrized Tests Not Running

**Symptom**: Parametrized tests (e.g., `@pytest.mark.parametrize`) are skipped or not found

**Solution**: Ensure fixture files exist and are properly formatted:

```bash
# Check fixture exists
ls tests/fixtures/mock/synthesizer_cases.jsonl

# Validate JSONL format (each line must be valid JSON)
cat tests/fixtures/mock/synthesizer_cases.jsonl | jq .
```

**Common issues**:
- Missing fixture file
- Invalid JSON in fixture file
- Incorrect fixture path in `load_jsonl()` call

---

### Slow Test Runs

**Symptom**: Test suite takes too long to run

**Solution 1**: Run only unit tests (fast):
```bash
pytest tests/unit/  # ~20 seconds
```

**Solution 2**: Skip slow integration tests:
```bash
pytest tests/ -m "not slow"
```

**Solution 3**: Run tests in parallel (unit tests only):
```bash
pytest tests/unit/ -n auto
```

**Solution 4**: Reduce batch size in integration tests:
```bash
# Edit test file to use smaller fixture
# e.g., vl-1.jsonl instead of vl.jsonl
```

---

### Coverage Report Not Generated

**Symptom**: `htmlcov/` directory not created after running with `--cov`

**Solution**: Install coverage plugin:
```bash
pip install pytest-cov
pytest tests/ --cov=autoqa --cov-report=html
```

**Verification**:
```bash
ls htmlcov/index.html
open htmlcov/index.html  # macOS
xdg-open htmlcov/index.html  # Linux
```

---

## CI/CD Integration

### GitHub Actions Example

```yaml
name: Test Suite
on: [push, pull_request]

jobs:
  unit-tests:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v3
      - uses: actions/setup-python@v4
        with:
          python-version: '3.12'
      - name: Install dependencies
        run: |
          pip install -r requirements.txt
          pip install pytest pytest-cov pytest-asyncio
      - name: Run unit tests
        run: pytest tests/unit/ --cov=autoqa --cov-report=xml
      - name: Upload coverage
        uses: codecov/codecov-action@v3
        with:
          file: ./coverage.xml

  integration-tests:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v3
      - uses: actions/setup-python@v4
        with:
          python-version: '3.12'
      - name: Install dependencies
        run: |
          pip install -r requirements.txt
          pip install pytest pytest-asyncio
      - name: Run integration tests
        env:
          PYTEST_API_KEY: ${{ secrets.PYTEST_API_KEY }}
          PYTEST_MODEL: gpt-4
          AUTOQA_FANOUT_CONCURRENCY: 2
        run: pytest tests/integration/ -m integration -v

  api-tests:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v3
      - uses: actions/setup-python@v4
        with:
          python-version: '3.12'
      - name: Install dependencies
        run: |
          pip install -r requirements.txt
          pip install pytest pytest-asyncio httpx
      - name: Run API tests
        env:
          PYTEST_API_KEY: ${{ secrets.PYTEST_API_KEY }}
          PYTEST_MODEL: gpt-4
        run: pytest tests/api/ -v
```

### GitLab CI Example

```yaml
stages:
  - test

unit-tests:
  stage: test
  image: python:3.12
  script:
    - pip install -r requirements.txt
    - pip install pytest pytest-cov pytest-asyncio
    - pytest tests/unit/ --cov=autoqa --cov-report=term

integration-tests:
  stage: test
  image: python:3.12
  variables:
    PYTEST_MODEL: "gpt-4"
    AUTOQA_FANOUT_CONCURRENCY: "2"
  script:
    - pip install -r requirements.txt
    - pip install pytest pytest-asyncio
    - pytest tests/integration/ -m integration -v
  only:
    - main
    - merge_requests
```

### Pre-commit Hook

Add to `.pre-commit-config.yaml`:

```yaml
repos:
  - repo: local
    hooks:
      - id: pytest-unit
        name: Run unit tests
        entry: pytest tests/unit/ -v
        language: system
        pass_filenames: false
        always_run: true
```

Install:
```bash
pip install pre-commit
pre-commit install
```

---

## Maintenance

### Adding New Tests

#### 1. Unit Tests

**Location**: `tests/unit/<pipeline>/`

**Template**:
```python
import pytest
from tests.helpers import make_mock_client
from autoqa.components.<pipeline>.nodes import make_<node>_node

async def test_<node>_<scenario>(sample_fixture):
    """Test description."""
    mock_response = '{"key": "value"}'
    node = make_<node>_node(make_mock_client(mock_response), "test-model")
    result = await node({"input": sample_fixture})
    assert result["output"] is not None
```

**Steps**:
1. Create test file in appropriate `tests/unit/<pipeline>/` directory
2. Add mock fixture to `tests/fixtures/mock/` if needed
3. Import required components
4. Write test function with descriptive name
5. Run test: `pytest tests/unit/<pipeline>/test_<node>.py -v`

#### 2. Integration Tests

**Location**: `tests/integration/<pipeline>/`

**Template**:
```python
import pytest

@pytest.mark.integration
async def test_<node>_integration(real_client, real_model, sample_fixture):
    """Test description."""
    node = make_<node>_node(real_client, real_model)
    result = await node({"input": sample_fixture})
    assert result["output"] is not None
```

**Steps**:
1. Create test file in appropriate `tests/integration/<pipeline>/` directory
2. Add `@pytest.mark.integration` decorator
3. Use `real_client` and `real_model` fixtures
4. Run test: `pytest tests/integration/<pipeline>/test_<node>.py -m integration -v`

#### 3. API Tests

**Location**: `tests/api/`

**Template**:
```python
import pytest
from fastapi import status

@pytest.mark.asyncio
async def test_<endpoint>_<scenario>(client):
    """Test description."""
    payload = {"key": "value"}
    response = await client.post("/api/v1/<endpoint>", json=payload)
    assert response.status_code == status.HTTP_200_OK
```

**Steps**:
1. Add test function to `tests/api/test_routes.py`
2. Use `client` fixture for HTTP requests
3. Run test: `pytest tests/api/test_routes.py::test_<endpoint>_<scenario> -v`

### Updating Mock Fixtures

**Location**: `tests/fixtures/mock/<node>_cases.jsonl`

**Schema**:
```json
{
  "id": "unique-test-case-id",
  "mock_response": "{...JSON response from LLM...}",
  "input_state": {...state passed to node...},
  "expected": {...expected output assertions...}
}
```

**Steps**:
1. Open fixture file: `tests/fixtures/mock/<node>_cases.jsonl`
2. Add new line with JSON object (no trailing comma)
3. Validate JSON: `cat tests/fixtures/mock/<node>_cases.jsonl | jq .`
4. Update test to use new fixture case

### Regenerating Fixtures

#### Gold Datasets

```bash
# Generate new gold dataset for RTM
python scripts/generate_gold_dataset.py --output tests/fixtures/gold/gold_dataset.jsonl

# Generate new gold dataset for TC
python scripts/generate_gold_dataset_tc.py --output tests/fixtures/gold/gold_dataset-tc.jsonl
```

#### Mock Responses

```bash
# Capture real LLM responses for mock fixtures
python scripts/capture_mock_responses.py \
  --node synthesizer \
  --output tests/fixtures/mock/synthesizer_cases.jsonl \
  --count 5
```

### Updating This Manual

**Location**: `docs/TEST_MANUAL.md`

**When to update**:
- New test added → Update Test Inventory table
- New fixture added → Update Fixture Reference table
- New environment variable → Update Environment Setup section
- New troubleshooting issue → Add to Troubleshooting section

**Steps**:
1. Edit `docs/TEST_MANUAL.md`
2. Update relevant section
3. Verify markdown formatting: `mdl docs/TEST_MANUAL.md`
4. Commit changes

---

## Appendix

### Test Markers

| Marker | Purpose | Usage |
|--------|---------|-------|
| `@pytest.mark.integration` | Integration tests (require real LLM) | `pytest -m integration` |
| `@pytest.mark.slow` | Slow tests (>1 minute) | `pytest -m "not slow"` to skip |
| `@pytest.mark.asyncio` | Async tests | Automatically detected by pytest-asyncio |
| `@pytest.mark.skip` | Skip test | `@pytest.mark.skip(reason="...")` |
| `@pytest.mark.parametrize` | Parametrized tests | `@pytest.mark.parametrize("arg", [val1, val2])` |

### Useful pytest Options

| Option | Purpose | Example |
|--------|---------|---------|
| `-v` | Verbose output | `pytest -v` |
| `-s` | Show print statements | `pytest -s` |
| `-x` | Stop on first failure | `pytest -x` |
| `--lf` | Run last failed tests | `pytest --lf` |
| `--ff` | Run failed tests first | `pytest --ff` |
| `-k` | Filter by test name | `pytest -k "synthesizer"` |
| `-m` | Filter by marker | `pytest -m integration` |
| `--collect-only` | List tests without running | `pytest --collect-only` |
| `--durations=10` | Show 10 slowest tests | `pytest --durations=10` |

### Test Fixtures (pytest)

| Fixture | Scope | Purpose | Defined In |
|---------|-------|---------|------------|
| `sample_requirement` | function | Sample Requirement object | `tests/conftest.py` |
| `sample_test_cases` | function | List of sample TestCase objects | `tests/conftest.py` |
| `sample_decomposed_requirement` | function | Sample DecomposedRequirement | `tests/conftest.py` |
| `sample_test_suite` | function | Sample TestSuite | `tests/conftest.py` |
| `sample_hazard` | function | Sample HazardRecord | `tests/conftest.py` |
| `real_client` | session | Real OpenAI client | `tests/conftest.py` |
| `real_model` | session | Model name from env | `tests/conftest.py` |
| `jsonl_recorders` | session | Input/output recorders for RTM | `tests/conftest.py` |
| `jsonl_recorders_tc` | session | Input/output recorders for TC | `tests/conftest.py` |
| `client` | function | Async HTTP client for API tests | `tests/api/test_routes.py` |

---

## Contact & Support

For questions or issues:
- **GitHub Issues**: [github.com/your-org/autoqa/issues](https://github.com/your-org/autoqa/issues)
- **Documentation**: [docs/](../docs/)
- **Slack**: #autoqa-support

---

**Last Updated**: 2024-01-XX  
**Version**: 1.0.0


# AutoQA Testing Guide

AutoQA provides three LangGraph-based reviewers — **Test Suite Reviewer** (RTM), **Test Case Reviewer**, and **Hazard Risk Reviewer** — each exposed as a compiled async pipeline and as a FastAPI endpoint. This guide covers how to run the key integration and API tests, what fixture files are available, what inputs each reviewer requires, and what output structures to expect.

---

## Prerequisites

```bash
# Install dependencies
uv sync

# Required environment variable
export OPENAI_API_KEY=<your-key>

# Optional overrides
export AUTOQA_MODEL=gpt-4o                  # default: gpt-4o
export AUTOQA_FANOUT_CONCURRENCY=5          # RTM parallel spec evaluators (default: 5)
                                             # TC reviewer uses 10 by default
```

Integration tests are marked `@pytest.mark.integration`. Run them with:

```bash
uv run pytest -m integration
```

---

## Fixture Files

Location: `tests/fixtures/external/`

| File | Rows | Coverage |
|------|------|----------|
| `test_suite_review_all_fields.jsonl` | 3 | REQ-PUMP-SW-042 (infusion rate limiting), REQ-EHR-AUTH-015 (MFA login), REQ-TELE-VIDEO-023 (adaptive video) — each with 4 test cases + 3 design docs |
| `test_case_review_all_fields.jsonl` | 3 | TC-PUMP-202 (watchdog fault injection), TC-EHR-AUTH-015-D (MFA bypass prevention), TC-TELE-VIDEO-023-C (video stabilization timer) — each with upstream requirements + design docs |
| `hazard_full_traceability.jsonl` | 1 | HAZ-PUMP-001 (over-infusion software loop hang) — catastrophic severity, full hierarchical traceability (user needs → system reqs → software reqs), test cases, and design docs |

Each file is newline-delimited JSON. Each line is a self-contained input object matching the corresponding reviewer's request schema.

---

## Integration Tests

These tests run the full compiled LangGraph pipeline end-to-end against the fixture files above. Each test loads all rows from its fixture, fans them out concurrently via `asyncio.gather`, and records inputs and outputs to `.jsonl` files in the run directory.

### Test Suite Reviewer

```bash
uv run pytest tests/integration/test_suite_reviewer/pipeline.py::test_test_suite_reviewer
```

**What it validates:**
- One `SynthesizedAssessment` produced per requirement row
- All 6 findings present (M1, M2, M3, M4, M5, R6)
- `overall_verdict=Yes` iff all M1–M5 are `"Yes"` or `"N-A"` (R6 does not affect overall verdict)
- `partial=True` only when `verdict="Yes"` and coverage is incomplete; `partial=False` on `"No"` and `"N-A"`

**Output files written:** `inputs.jsonl`, `outputs.jsonl` in the test run directory.

---

### Test Case Reviewer

```bash
uv run pytest tests/integration/test_case_reviewer/pipeline.py::test_case_suite_reviewer
```

**What it validates:**
- One `TestCaseAssessment` produced per test case row
- `evaluated_checklist` contains exactly 5 items with IDs:
  - `expected_result_support`
  - `expected_result_spec_align`
  - `test_case_achieves`
  - `test_case_logical_sequence`
  - `test_case_setup_clarity`
- `overall_verdict=Yes` iff all mandatory checklist objectives are `"Yes"`
- `expected_result_spec_align` verdict derived from spec coverage count: 0 covered → `("No", False)`, all covered → `("Yes", False)`, partial → `("Yes", True)`

**Output files written:** `inputs.jsonl`, `outputs.jsonl` in the test run directory.

---

### Hazard Risk Reviewer

```bash
uv run pytest tests/integration/hazard_risk_reviewer/pipeline.py::test_hazard_risk_reviewer
```

**What it validates:**
- One `RequirementReview` per traced requirement in the hazard record (expected: 6)
- Each `RequirementReview` contains a `SynthesizedAssessment` (M1–M5 + R6)
- `HazardAssessment` with exactly 7 findings (H1–H7)
- `overall_verdict=Yes` iff all H1–H7 are `"Yes"` or `"N-A"` — only H5 may be `"N-A"`
- H1–H4, H6–H7 verdicts must be `"Yes"` or `"No"` only

**Output files written:** `hazard_pipeline_state.json` (full graph state for manual inspection), `inputs.jsonl`, `outputs.jsonl`.

---

## API Happy-Path Tests

These tests spin up the FastAPI app via the test client and exercise the full request/response cycle for each endpoint.

### Test Suite Reviewer

```bash
uv run pytest tests/api/v1/test_test_suite_reviewer.py::test_rtm_review_happy_path
```

**Endpoint:** `POST /api/v1/review`

**Minimal payload:**
```json
{
  "thread_id": "test-thread-001",
  "requirement": {
    "req_id": "REQ-001",
    "text": "The system shall..."
  },
  "test_cases": [
    {
      "test_id": "TC-001",
      "description": "Verify that...",
      "setup": "Initialize system",
      "steps": "1. Do X\n2. Do Y",
      "expectedResults": "System responds with Z"
    }
  ],
  "design_docs": [
    {
      "doc_id": "DOC-001",
      "name": "Architecture Spec",
      "description": "..."
    }
  ]
}
```

**Validated in response:** HTTP 200, `synthesized_assessment` present, M1–M5 findings all present.

---

### Test Case Reviewer

```bash
uv run pytest tests/api/v1/test_test_case_reviewer.py::test_tc_review_happy_path
```

**Endpoint:** `POST /api/v1/test-case-review`

**Minimal payload:**
```json
{
  "thread_id": "tc-thread-001",
  "test_case": {
    "test_id": "TC-001",
    "description": "Verify that...",
    "setup": "...",
    "steps": "...",
    "expectedResults": "..."
  },
  "requirements": [
    {
      "req_id": "REQ-001",
      "text": "The system shall..."
    }
  ]
}
```

`design_docs` and `review_objectives` are optional. When `review_objectives` is omitted, the five standard objectives from `review_objectives.yaml` are used.

**Validated in response:** HTTP 200, `aggregated_assessment` present, `evaluated_checklist` has 5 items.

---

### Hazard Risk Reviewer

```bash
uv run pytest tests/api/v1/hazard_risk_reviewer.py::test_hazard_review_happy_path
```

**Endpoint:** `POST /api/v1/hazard-review`

**Minimal payload:**
```json
{
  "thread_id": "haz-thread-001",
  "hazard": {
    "hazard_id": "HAZ-001",
    "hazardous_situation_id": "HS-001",
    "hazard": "Over-infusion",
    "hazardous_situation": "Patient receives incorrect dose",
    "function": "Drug delivery",
    "ots_software": "None",
    "hazardous_sequence_of_events": "...",
    "software_related_causes": "...",
    "harm": "Patient injury",
    "severity": "Catastrophic",
    "exploitability_pre_mitigation": "Probable",
    "probability_of_harm_pre_mitigation": "Probable",
    "initial_risk_rating": "Unacceptable",
    "risk_control_measures": "...",
    "demonstration_of_effectiveness": "...",
    "severity_of_harm_post_mitigation": "Catastrophic",
    "exploitability_post_mitigation": "Remote",
    "probability_of_harm_post_mitigation": "Remote",
    "final_risk_rating": "Acceptable",
    "residual_risk_acceptability": "Acceptable",
    "requirements": [{"req_id": "REQ-001", "text": "..."}],
    "test_cases": [{"test_id": "TC-001", "description": "..."}],
    "design_docs": [],
    "user_needs": [],
    "system_requirements": []
  }
}
```

**Validated in response:** HTTP 200, `hazard_assessment` present with 7 H-code findings, `requirement_reviews` length matches traced requirements.

---

## Input Schema Reference

### Test Suite Reviewer — `POST /api/v1/review`

| Field | Type | Required | Notes |
|-------|------|----------|-------|
| `thread_id` | `string` | Yes | Alphanumeric, dashes, underscores; max 100 chars |
| `requirement` | `Requirement` | Yes | `{req_id?: string, text: string}` |
| `test_cases` | `TestCase[]` | Yes | Max 1000 items |
| `design_docs` | `DesignDocument[]` | No | Enables R6 (Design Alignment) finding |

**`TestCase`:** `{test_id, description, setup?, steps?, expectedResults?, in_baseline?}`

**`DesignDocument`:** `{doc_id, name, description}`

---

### Test Case Reviewer — `POST /api/v1/test-case-review`

| Field | Type | Required | Notes |
|-------|------|----------|-------|
| `thread_id` | `string` | Yes | |
| `test_case` | `TestCase` | Yes | |
| `requirements` | `Requirement[]` | Yes | At least one required |
| `review_objectives` | `ReviewObjective[]` | No | Defaults to 5 standard objectives from YAML |
| `design_docs` | `DesignDocument[]` | No | |

**`ReviewObjective`:** `{id: string, description: string, mandatory: bool}`

---

### Hazard Risk Reviewer — `POST /api/v1/hazard-review`

| Field | Type | Required | Notes |
|-------|------|----------|-------|
| `thread_id` | `string` | Yes | |
| `hazard` | `HazardRecord` | Yes | See full field list below |

**`HazardRecord` fields:**

| Field | Type |
|-------|------|
| `hazard_id` | `string` |
| `hazardous_situation_id` | `string` |
| `hazard` | `string` |
| `hazardous_situation` | `string` |
| `function` | `string` |
| `ots_software` | `string` |
| `hazardous_sequence_of_events` | `string` |
| `software_related_causes` | `string` |
| `harm` | `string` |
| `severity` | `string` |
| `exploitability_pre_mitigation` | `string` |
| `probability_of_harm_pre_mitigation` | `string` |
| `initial_risk_rating` | `string` |
| `risk_control_measures` | `string` |
| `demonstration_of_effectiveness` | `string` |
| `severity_of_harm_post_mitigation` | `string` |
| `exploitability_post_mitigation` | `string` |
| `probability_of_harm_post_mitigation` | `string` |
| `final_risk_rating` | `string` |
| `residual_risk_acceptability` | `string` |
| `requirements` | `Requirement[]` — software requirements |
| `test_cases` | `TestCase[]` — verification tests |
| `design_docs` | `DesignDocument[]` |
| `user_needs` | `Requirement[]` |
| `system_requirements` | `Requirement[]` |

Additional optional traceability fields: `new_hs_reference`, `sw_fmea_trace`, `sra_link`, `urra_item`, `harm_severity_rationale`.

#### Batch Route — `POST /api/v1/hazard-review/from-excel`

```json
{
  "thread_id_prefix": "batch-run-01",
  "file_path": "/abs/path/to/sha_table.xlsx",
  "sheet_name": "SHA Table"
}
```

---

## Expected Output Structures

### Test Suite Reviewer — `ReviewResponse`

```json
{
  "status": "completed",
  "thread_id": "...",
  "synthesized_assessment": {
    "requirement": {"req_id": "REQ-001", "text": "..."},
    "overall_verdict": "Yes | No",
    "mandatory_findings": [
      {
        "code": "M1 | M2 | M3 | M4 | M5 | R6",
        "dimension": "Functional | Negative | Boundary | Spec Coverage | Terminology | Design Alignment",
        "verdict": "Yes | No | N-A",
        "partial": false,
        "rationale": "One sentence.",
        "cited_test_case_ids": ["TC-001"],
        "uncovered_spec_ids": []
      }
    ],
    "comments": "...",
    "clarification_questions": []
  },
  "coverage_analysis": [
    {
      "spec_id": "SPEC-1",
      "covered_exists": true,
      "covered_by_test_cases": [
        {"test_case_id": "TC-001", "dimensions": ["functional"], "rationale": "..."}
      ]
    }
  ],
  "decomposed_requirement": {...},
  "test_suite": {...},
  "design_docs": [...]
}
```

---

### Test Case Reviewer — `TestCaseReviewResponse`

```json
{
  "status": "completed",
  "thread_id": "...",
  "aggregated_assessment": {
    "test_case": {...},
    "requirements": [...],
    "overall_verdict": "Yes | No",
    "evaluated_checklist": [
      {
        "id": "expected_result_support | expected_result_spec_align | test_case_achieves | test_case_logical_sequence | test_case_setup_clarity",
        "description": "...",
        "mandatory": true,
        "verdict": "Yes | No",
        "partial": false,
        "assessment": "..."
      }
    ],
    "comments": "...",
    "clarification_questions": []
  },
  "coverage_analysis": [...],
  "logical_structure_analysis": {"exists": true, "assessment": "..."},
  "prereqs_analysis": {"exists": true, "assessment": "..."},
  "decomposed_requirements": [...],
  "design_docs": [...]
}
```

---

### Hazard Risk Reviewer — `HazardReviewResponse`

```json
{
  "status": "completed",
  "thread_id": "...",
  "hazard": {<HazardRecord echoed from request>},
  "hazard_assessment": {
    "hazard_id": "HAZ-001",
    "overall_verdict": "Yes | No",
    "mandatory_findings": [
      {
        "code": "H1 | H2 | H3 | H4 | H5 | H6 | H7",
        "dimension": "<dimension name>",
        "verdict": "Yes | No | N-A",
        "rationale": "One sentence.",
        "cited_req_ids": ["REQ-001"],
        "cited_test_case_ids": ["TC-001"],
        "unblocked_items": []
      }
    ],
    "comments": "...",
    "clarification_questions": []
  },
  "requirement_reviews": [
    {
      "requirement": {"req_id": "REQ-001", "text": "..."},
      "synthesized_assessment": {<SynthesizedAssessment — same shape as RTM output>},
      "decomposed_requirement": {...},
      "test_suite": {...},
      "coverage_analysis": [...]
    }
  ]
}
```

---

## Verdict Logic Quick Reference

| Reviewer | `overall_verdict = "Yes"` when… | `partial = true` when… |
|----------|----------------------------------|------------------------|
| **Test Suite (RTM)** | All M1–M5 are `"Yes"` or `"N-A"` — R6 is excluded | `verdict="Yes"` but spec coverage is incomplete |
| **Test Case** | All **mandatory** checklist objectives are `"Yes"` | `verdict="Yes"` but not all specs covered (e.g., `expected_result_spec_align`) |
| **Hazard Risk** | All H1–H7 are `"Yes"` or `"N-A"` | N/A — hazard findings do not use `partial` |

**H-code N-A rule:** Only H5 (`Residual Risk Acceptability`) may return `"N-A"`. H1–H4, H6–H7 must be `"Yes"` or `"No"`.
